# Supermarket sales data set

## Set up block
*=> It must be executed minimum one time before executing next code blocks*

In [1]:
# === All imports ===
## global imports
import os
import sys
from itertools import groupby


import val
## spark imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date, udf, sum, count #avg, max, min, monotonically_increasing_id, round,
from pyspark.sql.types import StringType #, DoubleType, StructType, StructField, BooleanType

# === Get data souce file ===
data_file = "./data/Sample - Superstore.csv"
if os.path.exists(data_file):
    print(f"[ok]: data source file is found: {data_file}")
else:
    print(f"[ko]: data source file is missing: {data_file}\n/!\The following code blocks will not be able to function normally until this problem is resolved")
    sys.exit(1)

# === Create the Spark session
try:
    ## avoid the error: java.io.IOException: Cannot run program "python3"
    os.environ["PYSPARK_PYTHON"] = "python"
    os.environ["PYSPARK_DRIVER_PYTHON"] = "python"
    ## create spark session
    spark = SparkSession.builder.master("local[*]").appName("Supermarket sales data set").config("spark.ui.showConsoleProgress", "true").getOrCreate()
    print("[ok]: spark session created")
except Exception as e:
    print(f"[ko]: cannot create spark session: {e}\n/!\The following code blocks will not be able to function normally until this problem is resolved")
    sys.exit(1)

[ok]: data source file is found: ./data/Sample - Superstore.csv
[ok]: spark session created


## Part 1: Loading and Exploration
### 1.1) Load the CSV into a DataFrame with the header option

In [2]:
df_super_store = spark.read \
          .option("header", "true") \
          .option("inferSchema", "true") \
          .option("sep", ",") \
          .csv(data_file)

df_super_store.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

### 1.2) Display the DataFrame schema

In [3]:
df_super_store.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



### Cast the schema to avoid problems:
- Sales: string => Double
- Quantity: string => Integer
- Discount: string => Double

In [4]:
df_super_store = df_super_store.withColumn("Sales", col("Sales").cast("double"))
df_super_store = df_super_store.withColumn("Quantity", col("Quantity").cast("int"))
df_super_store = df_super_store.withColumn("Discount", col("Discount").cast("double"))
df_super_store.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



### 1.3) Display the first 20 rows

In [5]:
df_super_store.show(20)

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

In [6]:
total_nbr_of_rows = df_super_store.count()
print("Total number of rows: ", total_nbr_of_rows)

Total number of rows:  9994


### 1.5) Display the unique regions (column Region)

In [7]:
df_super_store.select(col("Region")).distinct().show()

+-------+
| Region|
+-------+
|  South|
|Central|
|   East|
|   West|
+-------+



## Part 2: Simple Transformations
### 2.1) Create a column Profit Margin = Profit / Sales

In [8]:
df_super_store_with_profit_m = df_super_store.withColumn("Profit Margin", col("Profit") / col("Sales"))
df_super_store_with_profit_m.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South

### 2.2) Create a column Year by extracting the year from Order Date

In [9]:
df_super_store_with_profit_m_with_y = df_super_store_with_profit_m.withColumn("Year", year(to_date(col("Order Date"),"M/d/yyyy")))
df_super_store_with_profit_m_with_y.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|Year|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|   

### 2.3) Create a column Total Value = Sales - Discount

In [10]:
df_super_store_with_profit_m_with_y_with_tot_value = df_super_store_with_profit_m_with_y.withColumn("Total Value", col("Sales") - col("Discount"))
df_super_store_with_profit_m_with_y_with_tot_value.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|Year|       Total Value|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gu

### 2.4) Display the first 10 rows with these new columns

In [11]:
df = df_super_store_with_profit_m_with_y_with_tot_value
df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|Year|       Total Value|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second C

### 2.5) Cache this DataFrame (you will reuse it several times)

In [12]:
df.cache()
df.count()

9994

## Part 3: UDF – Sales Categorization
### 3.1) Create a UDF categorizeSale that takes the Sales amount and returns:
- "Small sale" if < $100

- "Medium sale" if between 100 & 500

- "Large sale" if > $500

In [13]:
def categorize_sale(sales):
    # sales = float(sales)
    try:
        if sales is None:
            return "Unknown"
        if sales < 100:
            return "Small sale"
        elif 100 <= sales <= 500:
            return "Medium sale"
        else:# sales > 500:
            return "Large sale"
    except Exception as err:
        print("Error in categorize_sale()", err)

categorizeSale = udf(categorize_sale, StringType())

### 3.2) Apply this UDF to create a column Sale Category

In [14]:
dfWithUdf = df.withColumn("Sale Category", categorizeSale(col("Sales")))

dfWithUdf.show(5, truncate=False)
## put dfWithUdf in the cache
# dfWithUdf.cache()
# dfWithUdf.count()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+-------------------+----+-----------------+-------------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                               |Sales   |Quantity|Discount|Profit  |Profit Margin      |Year|Total Value      |Sale Category|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+-----

### 3.3) Display a few rows with this new column

In [15]:
dfWithUdf.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+-------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|Year|       Total Value|Sale Category|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+-------------+
|     1|CA-2016-152156| 11/8/2016|11/11/201

### 3.4) Count the number of sales per category (Small/Medium/Large)

In [16]:
dfWithUdf.groupBy("Sale Category").count().show()


+-------------+-----+
|Sale Category|count|
+-------------+-----+
|      Unknown|  300|
|   Large sale| 1151|
|  Medium sale| 2553|
|   Small sale| 5990|
+-------------+-----+



## Part 4: UDF – Discount Level
### 4.1) Create a UDF discountLevel that takes Discount and returns:
- "No discount" if = 0
- "Low discount" if between 0 and 0.2
- "High discount" if > 0.2

In [17]:
def discount_level(discount):
    try:
        if discount is None:
            return "Unknown"
        if discount == 0:
            return "No discount"
        elif 0 <= discount <= 0.2:
            return "Low discount"
        else:# > 0.2
            return "High discount"
    except Exception as err:
        print("Error in discount_level()", err)

discountLevel = udf(discount_level, StringType())

### 4.2) Apply this UDF to create a column Discount Level

In [18]:
dfWithUdf = df.withColumn("Discount Level", discountLevel(col("Discount")))
dfWithUdf.show(10)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+--------------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|      Profit Margin|Year|       Total Value|Discount Level|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+-------------------+----+------------------+--------------+
|     1|CA-2016-152156| 11/8/2016|11/11/

### 4.3) Calculate total revenue by discount level

In [19]:
dfWithUdf.groupby("Discount Level").agg(
    sum("Sales").alias("Total revenue(total sales)"),
).show()

+--------------+--------------------------+
|Discount Level|Total revenue(total sales)|
+--------------+--------------------------+
|       Unknown|                      NULL|
|   No discount|        1072777.3199999915|
|  Low discount|          838235.312500003|
| High discount|        361437.22379999945|
+--------------+--------------------------+



## Part 5: Basic Aggregations
### 5.1) Calculate total revenue (Sales) by region

In [20]:
dfWithUdf.groupby("Region").agg(
    sum("Sales").alias("Total revenue(total sales)"),
).show()

+-------+--------------------------+
| Region|Total revenue(total sales)|
+-------+--------------------------+
|  South|        388983.58500000037|
|Central|         497800.8728000007|
|   East|         672194.0539999981|
|   West|         713471.3445000004|
+-------+--------------------------+



### 5.2) Calculate total profit by product category (Category)

In [21]:
dfWithUdf.groupby("Category").agg(
    sum("Profit").alias("Total profit"),
).show()

+---------------+------------------+
|       Category|      Total profit|
+---------------+------------------+
|Office Supplies|120632.87839999991|
|      Furniture| 19686.42720000003|
|     Technology|145388.29659999989|
+---------------+------------------+



### 5.3) Calculate the number of orders by customer segment (Segment)

In [22]:
dfWithUdf.groupby("Segment").agg(
    count("Order ID").alias("Total orders"),
).show()

+-----------+------------+
|    Segment|Total orders|
+-----------+------------+
|   Consumer|        5191|
|Home Office|        1783|
|  Corporate|        3020|
+-----------+------------+



### 5.4) Identify the top 10 products (Product Name) by quantity sold

In [23]:
df_top_10 = dfWithUdf.groupBy("Product Name").agg(sum("Quantity").alias("Total quantity")).orderBy(col("Total quantity").desc()).limit(10)
df_top_10.show()

+--------------------+--------------+
|        Product Name|Total quantity|
+--------------------+--------------+
|"Tennsco Stur-D-S...|          4381|
|"Tenex 46"" x 60"...|          2264|
|"Rubbermaid Clust...|          2061|
|"Belkin 19"" Vent...|          1395|
|Wilson Jones Ledg...|          1209|
|"Tyvek Interoffic...|           935|
|"Belkin 19"" Cent...|           846|
|"Wilson Jones Ell...|           830|
|"Xerox Color Copi...|           591|
|"Eldon Delta Tria...|           543|
+--------------------+--------------+



### 5.5) Identify the top 5 states (State) with the highest revenue

In [24]:
df_top_5_ca_state = dfWithUdf.groupBy("State").agg(sum("Sales").alias("ca_total_state"))
df_top_5_ca_state.orderBy(col("ca_total_state").desc()).limit(5).show()

+------------+------------------+
|       State|    ca_total_state|
+------------+------------------+
|  California| 450567.5915000007|
|    New York|309453.63299999974|
|       Texas|169553.63180000003|
|  Washington|136590.17199999996|
|Pennsylvania|114911.23700000002|
+------------+------------------+



## Part 6: Broadcast Variable – Region Codes
### 6.1) Create a Map that associates each region with a code:

In [25]:
regionCodes = {
  "East": "EST",
  "West": "WST",
  "Central": "CTR",
  "South": "STH"
}
#Broadcast de la Map
broadcastRegionCodes = spark.sparkContext.broadcast(regionCodes)